In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
    
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/justinas/housing-in-london/housing_in_london_monthly_variables.csv
/kaggle/input/datasets/justinas/housing-in-london/housing_in_london_yearly_variables.csv


In [26]:
path = '/kaggle/input/datasets/justinas/housing-in-london/housing_in_london_monthly_variables.csv'
df = pd.read_csv(path)

In [27]:
df

,date,area,average_price,code,houses_sold,no_of_crimes,borough_flag
0,1995-01-01,city of london,91449,E09000001,17.0,NaN,1
1,1995-02-01,city of london,82203,E09000001,7.0,NaN,1
2,1995-03-01,city of london,79121,E09000001,14.0,NaN,1
3,1995-04-01,city of london,77101,E09000001,7.0,NaN,1
4,1995-05-01,city of london,84409,E09000001,10.0,NaN,1
...,...,...,...,...,...,...,...
13544,2019-09-01,england,249942,E92000001,64605.0,NaN,0
13545,2019-10-01,england,249376,E92000001,68677.0,NaN,0
13546,2019-11-01,england,248515,E92000001,67814.0,NaN,0
13547,2019-12-01,england,250410,E92000001,NaN,NaN,0


### Checking data

In [28]:
print("1. Checking shape (rows and columns)")
print(df.shape)

print("\n2. Checking Info (datatypes and null/non-null counts)")
print(df.info())

print("\n3. Checking for missing values")
print(df.isna().sum())

print("\n4. Checking 5 rows")
print(df.head())

print("\n5. Checking unique areas (first 15)")
print(df['area'].unique()[:15])

print("\n6. Checking borough_flag unique values")
print(df['borough_flag'].unique())

1. Checking shape (rows and columns)
(13549, 7)

2. Checking Info (datatypes and null/non-null counts)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13549 entries, 0 to 13548
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           13549 non-null  object 
 1   area           13549 non-null  object 
 2   average_price  13549 non-null  int64  
 3   code           13549 non-null  object 
 4   houses_sold    13455 non-null  float64
 5   no_of_crimes   7439 non-null   float64
 6   borough_flag   13549 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 741.1+ KB
None

3. Checking for missing values
date                0
area                0
average_price       0
code                0
houses_sold        94
no_of_crimes     6110
borough_flag        0
dtype: int64

4. Checking 5 rows
         date            area  average_price       code  houses_sold  \
0  1995-01-01  city of london    

### Data cleaning & Type conversion

In [30]:
# Convert 'date' to actual datetime objects
df['date'] = pd.to_datetime(df['date'])

# Standardize text: Make area names Title Case (e.g., "city of london" -> "City of London")
df['area'] = df['area'].str.title()

# Optimize Memory: Convert low-cardinality text and binary columns to 'category'
df['area'] = df['area'].astype('category')
df['code'] = df['code'].astype('category')
df['borough_flag'] = df['borough_flag'].astype('category')

# Handle Missing Values
# For houses_sold (only 94 missing), fill with the median of the column
median_houses_sold = df['houses_sold'].median()
df['houses_sold'] = df['houses_sold'].fillna(median_houses_sold)

# For no_of_crimes (6110 missing), fill with 0. 
# Assuming since there is no data about no_of_crimes, they must be 0
df['no_of_crimes'] = df['no_of_crimes'].fillna(0)

# Verify the changes
print("1. Updated Info")
print(df.info())

print("\n2. Updated Missing Values")
print(df.isna().sum())

print("\n3. First 5 rows")
print(df.head())

1. Updated Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13549 entries, 0 to 13548
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           13549 non-null  datetime64[ns]
 1   area           13549 non-null  category      
 2   average_price  13549 non-null  int64         
 3   code           13549 non-null  category      
 4   houses_sold    13549 non-null  float64       
 5   no_of_crimes   13549 non-null  float64       
 6   borough_flag   13549 non-null  category      
dtypes: category(3), datetime64[ns](1), float64(2), int64(1)
memory usage: 466.1 KB
None

2. Updated Missing Values
date             0
area             0
average_price    0
code             0
houses_sold      0
no_of_crimes     0
borough_flag     0
dtype: int64

3. First 5 rows
        date            area  average_price       code  houses_sold  \
0 1995-01-01  City Of London          91449  E09000001         17.0

### Feature Engineering for ML

In [31]:
# 1. Extract Temporal Features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name().astype('category')

# 2. Create a Binary Feature
median_crimes = df['no_of_crimes'].median()
# Compare -> Boolean Series -> Integers -> Category
df['is_high_crime'] = (df['no_of_crimes'] > median_crimes).astype(int).astype('category')

# 3. Rename columns for better ML pipeline readability
df = df.rename(columns={
    'average_price': 'avg_price',
    'houses_sold': 'houses_sold_count',
    'borough_flag': 'is_borough'
})

# 4. Verify the new structure
print("1. New Dataframe info")
print(df.info())

print("\n2. First 10 rows with new features")
cols_to_show = ['date', 'year', 'month', 'month_name', 'avg_price', 'no_of_crimes', 'is_high_crime', 'is_borough']
print(df[cols_to_show].head(10))

1. New Dataframe info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13549 entries, 0 to 13548
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               13549 non-null  datetime64[ns]
 1   area               13549 non-null  category      
 2   avg_price          13549 non-null  int64         
 3   code               13549 non-null  category      
 4   houses_sold_count  13549 non-null  float64       
 5   no_of_crimes       13549 non-null  float64       
 6   is_borough         13549 non-null  category      
 7   year               13549 non-null  int32         
 8   month              13549 non-null  int32         
 9   month_name         13549 non-null  category      
 10  is_high_crime      13549 non-null  category      
dtypes: category(5), datetime64[ns](1), float64(2), int32(2), int64(1)
memory usage: 599.0 KB
None

2. First 10 rows with new features
        date  year 

### Export cleaned data

In [32]:
# Saving as Parquet (as it preserves the datatypes and compressed better meaning smaller size)
parquet_path = '/kaggle/working/cleaned_london_housing.parquet'
df.to_parquet(parquet_path, index=False, engine='pyarrow')
print(f"Saved Parquet to: {parquet_path}")

# Saving as CSV (Universal fallback)
csv_path = '/kaggle/working/cleaned_london_housing.csv'
df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")

Saved Parquet to: /kaggle/working/cleaned_london_housing.parquet
Saved CSV to: /kaggle/working/cleaned_london_housing.csv
